# Solutions — Column Operations

**Dataset:** `samples.bakehouse.sales_transactions`, `samples.tpch.orders`, `samples.wanderbricks.bookings`

**Topics:** withColumn, cast, withColumnRenamed, drop, when/otherwise, lit, expr

> Reference solutions for practice notebooks.
> **Try the problem yourself first** — only open this when you've made a genuine attempt.
>
> Each solution includes a `# Why:` comment explaining the approach chosen.

In [ ]:
from pyspark.sql import functions as F, types as T
transactions = spark.table("samples.bakehouse.sales_transactions")
orders       = spark.table("samples.tpch.orders")
bookings     = spark.table("samples.wanderbricks.bookings")

---
## Problem 1 — withColumn and cast

In [ ]:
# Why: T.StringType() and T.DoubleType() are explicit — avoids relying on string shortcuts.
result_1 = (
    transactions.withColumn("card_str",       F.col("cardNumber").cast(T.StringType()))
                .withColumn("revenue_double", F.col("totalPrice").cast(T.DoubleType()))
)

---
## Problem 2 — Rename and Drop Columns

In [ ]:
# Why: chain withColumnRenamed for each rename, then drop unwanted columns at the end.
result_2 = (
    orders.withColumnRenamed("o_orderkey",  "order_id")
          .withColumnRenamed("o_custkey",   "customer_id")
          .withColumnRenamed("o_totalprice","total_price")
          .withColumnRenamed("o_orderdate", "order_date")
          .withColumnRenamed("o_orderstatus","status")
          .withColumnRenamed("o_orderpriority","priority")
          .withColumnRenamed("o_shippriority","ship_priority")
          .drop("o_comment", "o_clerk")
)

---
## Problem 3 — Conditional Column with when/otherwise

In [ ]:
# Why: Spark evaluates when() top-to-bottom and stops at first match,
# so the second condition only fires when totalPrice >= 20.
result_3 = (
    transactions.withColumn("price_category",
        F.when(F.col("totalPrice") < 20, "Low")
         .when(F.col("totalPrice") <= 100, "Medium")
         .otherwise("High")
    ).groupBy("price_category")
     .agg(
         F.count("*").alias("transaction_count"),
         F.round(F.avg("totalPrice"), 2).alias("avg_price")
     )
)

---
## Problem 4 — Literal and Expression Columns

In [ ]:
# Why: F.lit() creates a constant column; F.expr() lets you write SQL-style expressions.
# Guard against division by zero for per_night_price.
result_4 = (
    bookings.withColumn("platform",       F.lit("Wanderbricks"))
            .withColumn("stay_nights",    F.expr("datediff(check_out, check_in)"))
            .withColumn("per_night_price",
                F.when(F.col("stay_nights") > 0,
                       F.round(F.col("total_price") / F.col("stay_nights"), 2))
                 .otherwise(None))
            .select("booking_id", "platform", "check_in", "check_out",
                    "stay_nights", "total_price", "per_night_price")
)

---
## Problem 5 — Select with Expressions

In [ ]:
# Why: a single select() is more efficient than multiple withColumn calls —
# it builds one logical plan step instead of many.
result_5 = transactions.select(
    F.col("transactionID").alias("transaction_id"),
    F.to_date("dateTime").alias("transaction_date"),
    F.col("product"),
    F.col("quantity").alias("qty"),
    F.col("unitPrice").alias("unit_price"),
    F.round(F.col("totalPrice"), 2).alias("total_price")
)